# P19 ClaimIQ - Week 4: Source-to-Bronze Ingestion

### ZENAIZ x BVRIT Hyderabad Data Engineering Internship

**Project:** P19 ClaimIQ
**Notebook:** `notebooks/02_bronze_ingestion.ipynb`
**Learning approach:** Spark SQL first
**Week 4 scope:** the six approved ClaimIQ batch source files to Bronze Delta tables

This notebook applies the Week 4 Source-to-Bronze method from the PageLoop
worked example to the actual approved ClaimIQ project files:
`claims.parquet`, `policies.json`, `policyholders.csv`, `products.csv`,
`providers.csv` and `claim_payments.csv`.

The six `claim_status_event.json` streaming drops referenced in the Project
Playbook are **not** part of Week 4 - they belong to the Week 10 streaming
work - and are intentionally excluded from this notebook.

Read the instruction above each code cell, run one cell at a time, inspect the
result, and complete the checkpoint before continuing.

## How to use this notebook

This is an executable Databricks lab, not a reading-only document.

1. Read the short explanation above a code cell.
2. Check that the path, filename and table name are correct.
3. Run only that code cell.
4. Inspect the Databricks result before moving forward.
5. Compare the result with the stated expected observation.
6. Complete the checkpoint or record the required evidence.

Do not use **Run all** on the first attempt. Run from top to bottom, one cell
at a time. After the notebook has completed successfully once, you may use
**Run all** for the controlled repeat-run test.

## Databricks cell and syntax rules

| Cell type | Purpose | Rule used in this notebook |
|---|---|---|
| Markdown | Instructions, concepts and checkpoints | Read before running the next cell |
| `%sql` | Source views, Delta tables and validation | Main Week 4 implementation language |
| `%fs` | Volume file listing | Used only for the file-level visibility check |
| `%python` | Schema/column inspection only | Minimal PySpark, no business logic |

SQL formatting conventions used throughout:

- SQL keywords are written in uppercase.
- One selected column is placed on each line when the list is long.
- Two-space indentation is used inside SQL clauses.
- Temporary source views end with `_source`.
- Bronze table names begin with `bronze_claimiq_`.
- Technical metadata columns begin with an underscore.
- Business columns keep the approved source names and values (raw-preserving).

## Week 4 objectives

By the end of this notebook you should be able to:

1. explain why a Bronze layer is required for ClaimIQ;
2. identify every approved ClaimIQ batch source;
3. read Parquet, JSON Lines and CSV files from a Unity Catalog Volume;
4. create one persistent Bronze Delta table per approved batch source;
5. preserve source business values without cleaning or transforming them;
6. add ingestion, lineage, schema-version and record-hash metadata;
7. reconcile source and Bronze record counts for all six sources;
8. prove a repeat run does not create unintended duplicate records;
9. inspect the six Bronze tables and lineage in Catalog Explorer;
10. record Week 4 evidence in the repository and explain the work in a mentor review.

## ClaimIQ source-to-Bronze plan

| Business dataset | Approved source file | Format | Business key | Bronze target |
|---|---|---|---|---|
| Claims | `claims.parquet` | Parquet (Snappy) | `claim_id` | `bronze_claimiq_claims` |
| Policies | `policies.json` | JSON Lines | `policy_id` | `bronze_claimiq_policies` |
| Policyholders | `policyholders.csv` | CSV | `policyholder_id` | `bronze_claimiq_policyholders` |
| Products | `products.csv` | CSV | `product_id` | `bronze_claimiq_products` |
| Providers | `providers.csv` | CSV | `provider_id` | `bronze_claimiq_providers` |
| Claim payments | `claim_payments.csv` | CSV | `payment_id` | `bronze_claimiq_claim_payments` |

**Join warning (from the Project Playbook):** one policyholder can hold many
policies; one policy can produce many claims; one claim can have many
payments and events. Never row-level join payments to claims and then count
claim rows in this notebook or any later week - Bronze keeps every source at
its own physical grain.

Week 4 does **not** clean, standardise, join or deduplicate business values.
That begins in Week 5 (Silver Candidate transformation).

# Part 1 - Prepare Databricks

Before running the notebook:

1. import this notebook into Databricks;
2. attach Serverless notebook compute;
3. upload the six approved ClaimIQ batch files to
   `/Volumes/workspace/default/claimiq`;
4. keep the filenames unchanged (`claims.parquet`, `policies.json`,
   `policyholders.csv`, `products.csv`, `providers.csv`,
   `claim_payments.csv`);
5. run the notebook from the first code cell downward.

If your workspace uses a different approved catalog, schema or Volume path,
change every path and SQL context in this notebook consistently before
continuing.

## 1.1 Select the catalog and schema

The ClaimIQ project uses the `workspace` catalog and `default` schema. The
final query confirms the active location.

**Continue when:** the result shows `workspace` and `default`. Otherwise,
correct the approved names before creating any view or table.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;

## 1.2 Confirm the files in the Volume

This file-system cell checks whether Databricks can see the uploaded ClaimIQ
files. It does not read their contents.

Confirm that the listing includes all six approved batch files:
`claims.parquet`, `policies.json`, `policyholders.csv`, `products.csv`,
`providers.csv`, `claim_payments.csv`. If one file is missing or has a
different name, correct the upload before running any ingestion cell.

In [ ]:
%fs
ls /Volumes/workspace/default/claimiq

## 1.3 Confirm the Week 3 handoff

Week 4 assumes that Week 3 profiling succeeded. Before moving ahead, confirm:

- all six files opened in Week 3;
- their headers/field names matched the approved Data Dictionary;
- their source counts were recorded from Databricks;
- unexplained schema, format or grain problems were resolved;
- anti-join and cardinality risks between claims, payments and policies were
  understood (one-to-many everywhere downstream of `claim_id`).

Do not use Week 4 ingestion to hide a Week 3 source problem.

# Part 2 - Understand the Bronze contract

A Bronze table contains:

1. all approved business columns from the source, unmodified; and
2. technical columns added by the ingestion process.

Week 4 does not clean, cast, standardise or reinterpret source business
values - that is Silver Candidate work from Week 5 onward.

## 2.1 Why Bronze contains `_` technical fields

| Technical field | Plain-language meaning | Question it answers |
|---|---|---|
| `_source_file_name` | Filename that supplied the row | Which batch file did this row come from? |
| `_source_file_path` | Exact Volume location read by the notebook | Where was the file stored? |
| `_ingested_at` | Timestamp when the Bronze-ready row was prepared | When was this load performed? |
| `_ingestion_run_id` | Controlled identifier shared by one load | Which Week 4 run produced the row? |
| `_schema_version` | Version of the approved source contract | Which field definition was used? |
| `_record_hash` | SHA-256 fingerprint of the business values | Did the row content remain the same across reruns? |
| `_rescued_payload` | Parser context retained for an unreadable record | Did Spark encounter content it could not map normally? |

These fields support traceability. They do not clean, enrich or score the
business data, and the Playbook's minimum required set - `source_file`,
`batch_id`, `ingestion_timestamp`, `record_hash` - is fully covered by them
(`_ingestion_run_id` plays the role of the batch identity used for the
rerun-safety proof in Part 6).

## 2.2 Controlled run identity

Every row written in this run carries the same `_ingestion_run_id` and
`_schema_version`, produced once and reused for all six sources so that a
single load can always be identified and reconciled as one unit.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW week4_run_control AS
SELECT
  'W04_CLAIMIQ_RUN01' AS ingestion_run_id,
  'claimiq_source_v1.0' AS schema_version;

In [ ]:
%sql
SELECT * FROM week4_run_control;

# Part 3 - Build the claims Bronze table (detailed worked example)

`claims.parquet` is the anchor entity of ClaimIQ and is used here as the
detailed worked example. The remaining five sources in Part 4 and Part 5
follow the identical method with fewer repeated explanations.

## 3.1 Read the claims source

**Purpose:** expose `claims.parquet` as a queryable source view without
changing the file.

**How it works:**

- Parquet is self-describing, so no explicit `CREATE TEMP VIEW ( ... )`
  column list or `_corrupt_record` handling is required the way it is for
  CSV/JSON - Spark reads the embedded schema and physical types as written.
- The path points directly at the approved Volume file.

**What to inspect:** column names and types match the approved Data
Dictionary for `claims.parquet` (`claim_id`, lifecycle timestamps, amount
columns, `review_flag` as boolean, etc.).

**When to continue:** the row count matches your Week 3 source count for
`claims.parquet`.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_source
USING PARQUET
OPTIONS (
  path '/Volumes/workspace/default/claimiq/claims.parquet'
);

In [ ]:
%sql
SELECT *
FROM claims_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE claims_source;

In [ ]:
%python
claims_source_df = spark.table("claims_source")
claims_source_df.printSchema()
print("Columns:", claims_source_df.columns)

In [ ]:
%sql
SELECT COUNT(*) AS claims_source_count
FROM claims_source;

## 3.2 Add lineage metadata and build the Bronze-ready view

**Purpose:** attach ingestion lineage to every claim row before it is
persisted, without altering any business value.

**How it works:**

- `CROSS JOIN week4_run_control` attaches the same `_ingestion_run_id` and
  `_schema_version` to every row in this load.
- `_record_hash` is a SHA-256 fingerprint over every business column so a
  future rerun can prove the row content did not silently change.
- Parquet has no `_corrupt_record` column, so `_rescued_payload` is set to
  `NULL` for this source by contract, not by omission.

**What to inspect:** every business column from `claims_source` is present
unchanged, alongside the seven technical columns.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_with_metadata AS
SELECT
  c.*,
  'claims.parquet' AS _source_file_name,
  '/Volumes/workspace/default/claimiq/claims.parquet' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version
FROM claims_source c
CROSS JOIN week4_run_control r;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_bronze_ready AS
SELECT
  c.*,
  sha2(concat_ws('||',
    coalesce(cast(c.source_record_id AS STRING), '<NULL>'),
    coalesce(cast(c.claim_id AS STRING), '<NULL>'),
    coalesce(cast(c.policy_id AS STRING), '<NULL>'),
    coalesce(cast(c.policyholder_id AS STRING), '<NULL>'),
    coalesce(cast(c.product_id AS STRING), '<NULL>'),
    coalesce(cast(c.provider_id AS STRING), '<NULL>'),
    coalesce(cast(c.claim_type AS STRING), '<NULL>'),
    coalesce(cast(c.loss_category AS STRING), '<NULL>'),
    coalesce(cast(c.loss_date AS STRING), '<NULL>'),
    coalesce(cast(c.submission_timestamp AS STRING), '<NULL>'),
    coalesce(cast(c.review_timestamp AS STRING), '<NULL>'),
    coalesce(cast(c.decision_timestamp AS STRING), '<NULL>'),
    coalesce(cast(c.settlement_timestamp AS STRING), '<NULL>'),
    coalesce(cast(c.closure_timestamp AS STRING), '<NULL>'),
    coalesce(cast(c.claim_status AS STRING), '<NULL>'),
    coalesce(cast(c.outcome_code AS STRING), '<NULL>'),
    coalesce(cast(c.requested_amount AS STRING), '<NULL>'),
    coalesce(cast(c.approved_amount AS STRING), '<NULL>'),
    coalesce(cast(c.reserve_amount AS STRING), '<NULL>'),
    coalesce(cast(c.deductible_amount AS STRING), '<NULL>'),
    coalesce(cast(c.currency_code AS STRING), '<NULL>'),
    coalesce(cast(c.risk_band AS STRING), '<NULL>'),
    coalesce(cast(c.review_flag AS STRING), '<NULL>'),
    coalesce(cast(c.exception_code AS STRING), '<NULL>'),
    coalesce(cast(c.source_system AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  CAST(NULL AS STRING) AS _rescued_payload
FROM claims_with_metadata c;

In [ ]:
%sql
SELECT
  source_record_id,
  claim_id,
  claim_status,
  _source_file_name,
  _ingestion_run_id,
  _schema_version,
  _record_hash,
  _rescued_payload
FROM claims_bronze_ready
LIMIT 10;

## 3.3 Persist the claims Bronze table

**Purpose:** turn the temporary Bronze-ready view into the durable Bronze
Delta asset for the claims source.

**How it works:** `CREATE OR REPLACE TABLE ... AS SELECT` writes a full,
deterministic snapshot of this controlled run. Because it replaces rather
than appends, rerunning this cell with the same source file cannot double
the batch - see Part 6 for the proof.

**Continue when:** `bronze_claimiq_claims` is visible in Catalog Explorer
under `workspace.default`.

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_claimiq_claims
USING DELTA
AS
SELECT *
FROM claims_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_claimiq_claims;

In [ ]:
%sql
DESCRIBE TABLE bronze_claimiq_claims;

In [ ]:
%sql
SELECT *
FROM bronze_claimiq_claims
LIMIT 10;

## 3.4 Reconcile claims source and Bronze counts

**Checkpoint:** `source_count` must equal `bronze_count`. If it does not,
stop and correct the read options or path before continuing to Part 4.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM claims_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_claimiq_claims) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM claims_source)
       = (SELECT COUNT(*) FROM bronze_claimiq_claims)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

# Part 4 - Build the policies Bronze table (JSON worked example)

`policies.json` is newline-delimited JSON (JSON Lines), which needs a
different reader configuration from Parquet or CSV. It is shown here in
full detail once; Part 5 applies the same CSV method to the three remaining
sources with fewer repeated explanations.

## 4.1 Read the policies source

**Purpose:** expose `policies.json` as a queryable source view with an
explicit STRING schema, so numeric- and date-looking JSON values are
preserved exactly as written rather than silently coerced by schema
inference.

**How it works:**

- `multiLine 'false'` tells Spark this is JSON Lines (one JSON object per
  line), not one large JSON document.
- `mode 'PERMISSIVE'` with `columnNameOfCorruptRecord '_corrupt_record'`
  routes any line Spark cannot parse into a rescued column instead of
  silently dropping it.

**What to inspect:** all 14 approved policy fields are present as STRING,
plus `_corrupt_record`.

**When to continue:** the row count matches your Week 3 source count for
`policies.json`.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policies_source
(
  source_record_id STRING,
  policy_id STRING,
  policyholder_id STRING,
  product_id STRING,
  coverage_type STRING,
  policy_start_date STRING,
  policy_end_date STRING,
  coverage_limit STRING,
  deductible_amount STRING,
  premium_amount STRING,
  currency_code STRING,
  policy_status STRING,
  region_code STRING,
  source_system STRING,
  _corrupt_record STRING
)
USING JSON
OPTIONS (
  path '/Volumes/workspace/default/claimiq/policies.json',
  multiLine 'false',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM policies_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE policies_source;

In [ ]:
%python
policies_source_df = spark.table("policies_source")
policies_source_df.printSchema()
print("Columns:", policies_source_df.columns)

In [ ]:
%sql
SELECT COUNT(*) AS policies_source_count
FROM policies_source;

## 4.2 Add lineage metadata and build the Bronze-ready view

Same method as claims: attach the shared run control, fingerprint the
business columns, and retain any rescued payload.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policies_with_metadata AS
SELECT
  p.*,
  'policies.json' AS _source_file_name,
  '/Volumes/workspace/default/claimiq/policies.json' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version
FROM policies_source p
CROSS JOIN week4_run_control r;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policies_bronze_ready AS
SELECT
  p.* EXCEPT (_corrupt_record),
  sha2(concat_ws('||',
    coalesce(cast(p.source_record_id AS STRING), '<NULL>'),
    coalesce(cast(p.policy_id AS STRING), '<NULL>'),
    coalesce(cast(p.policyholder_id AS STRING), '<NULL>'),
    coalesce(cast(p.product_id AS STRING), '<NULL>'),
    coalesce(cast(p.coverage_type AS STRING), '<NULL>'),
    coalesce(cast(p.policy_start_date AS STRING), '<NULL>'),
    coalesce(cast(p.policy_end_date AS STRING), '<NULL>'),
    coalesce(cast(p.coverage_limit AS STRING), '<NULL>'),
    coalesce(cast(p.deductible_amount AS STRING), '<NULL>'),
    coalesce(cast(p.premium_amount AS STRING), '<NULL>'),
    coalesce(cast(p.currency_code AS STRING), '<NULL>'),
    coalesce(cast(p.policy_status AS STRING), '<NULL>'),
    coalesce(cast(p.region_code AS STRING), '<NULL>'),
    coalesce(cast(p.source_system AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(p._corrupt_record AS STRING) AS _rescued_payload
FROM policies_with_metadata p;

In [ ]:
%sql
SELECT
  source_record_id,
  policy_id,
  policy_status,
  _source_file_name,
  _ingestion_run_id,
  _schema_version,
  _record_hash,
  _rescued_payload
FROM policies_bronze_ready
LIMIT 10;

## 4.3 Persist the policies Bronze table and reconcile

**Continue when:** `bronze_claimiq_policies` is visible in Catalog Explorer
and `source_count` equals `bronze_count`.

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_claimiq_policies
USING DELTA
AS
SELECT *
FROM policies_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_claimiq_policies;

In [ ]:
%sql
SELECT *
FROM bronze_claimiq_policies
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM policies_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_claimiq_policies) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM policies_source)
       = (SELECT COUNT(*) FROM bronze_claimiq_policies)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

# Part 5 - Build the remaining CSV Bronze tables

`policyholders.csv`, `products.csv`, `providers.csv` and `claim_payments.csv`
all follow the identical CSV method already shown for reference sources in
the PageLoop example: an explicit all-STRING schema, `_corrupt_record`
rescue column, shared run-control metadata, a business-value hash, and a
`CREATE OR REPLACE TABLE` write. Each one is repeated below with the
minimum explanation needed to run it correctly.

## 5.1 Policyholders - `policyholders.csv` -> `bronze_claimiq_policyholders`

**Business key:** `policyholder_id`. **What to inspect after each cell:** column
names match the Data Dictionary; the source and Bronze counts reconcile;
technical metadata columns are fully populated.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policyholders_source
(
  source_record_id STRING,
  policyholder_id STRING,
  policyholder_segment STRING,
  age_band STRING,
  region_code STRING,
  risk_band STRING,
  join_date STRING,
  active_flag STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/claimiq/policyholders.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM policyholders_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE policyholders_source;

In [ ]:
%sql
SELECT COUNT(*) AS policyholders_source_count
FROM policyholders_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policyholders_with_metadata AS
SELECT
  s.*,
  'policyholders.csv' AS _source_file_name,
  '/Volumes/workspace/default/claimiq/policyholders.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version
FROM policyholders_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policyholders_bronze_ready AS
SELECT
  s.* EXCEPT (_corrupt_record),
  sha2(concat_ws('||',
    coalesce(cast(s.source_record_id AS STRING), '<NULL>'),
    coalesce(cast(s.policyholder_id AS STRING), '<NULL>'),
    coalesce(cast(s.policyholder_segment AS STRING), '<NULL>'),
    coalesce(cast(s.age_band AS STRING), '<NULL>'),
    coalesce(cast(s.region_code AS STRING), '<NULL>'),
    coalesce(cast(s.risk_band AS STRING), '<NULL>'),
    coalesce(cast(s.join_date AS STRING), '<NULL>'),
    coalesce(cast(s.active_flag AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM policyholders_with_metadata s;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_claimiq_policyholders
USING DELTA
AS
SELECT *
FROM policyholders_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_claimiq_policyholders;

In [ ]:
%sql
SELECT *
FROM bronze_claimiq_policyholders
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM policyholders_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_claimiq_policyholders) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM policyholders_source)
       = (SELECT COUNT(*) FROM bronze_claimiq_policyholders)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

## 5.2 Products - `products.csv` -> `bronze_claimiq_products`

**Business key:** `product_id`. **What to inspect after each cell:** column
names match the Data Dictionary; the source and Bronze counts reconcile;
technical metadata columns are fully populated.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW products_source
(
  source_record_id STRING,
  product_id STRING,
  product_name STRING,
  product_category STRING,
  coverage_type STRING,
  coverage_limit STRING,
  deductible_default STRING,
  sla_target_days STRING,
  provider_required_flag STRING,
  currency_code STRING,
  active_flag STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/claimiq/products.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM products_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE products_source;

In [ ]:
%sql
SELECT COUNT(*) AS products_source_count
FROM products_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW products_with_metadata AS
SELECT
  s.*,
  'products.csv' AS _source_file_name,
  '/Volumes/workspace/default/claimiq/products.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version
FROM products_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW products_bronze_ready AS
SELECT
  s.* EXCEPT (_corrupt_record),
  sha2(concat_ws('||',
    coalesce(cast(s.source_record_id AS STRING), '<NULL>'),
    coalesce(cast(s.product_id AS STRING), '<NULL>'),
    coalesce(cast(s.product_name AS STRING), '<NULL>'),
    coalesce(cast(s.product_category AS STRING), '<NULL>'),
    coalesce(cast(s.coverage_type AS STRING), '<NULL>'),
    coalesce(cast(s.coverage_limit AS STRING), '<NULL>'),
    coalesce(cast(s.deductible_default AS STRING), '<NULL>'),
    coalesce(cast(s.sla_target_days AS STRING), '<NULL>'),
    coalesce(cast(s.provider_required_flag AS STRING), '<NULL>'),
    coalesce(cast(s.currency_code AS STRING), '<NULL>'),
    coalesce(cast(s.active_flag AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM products_with_metadata s;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_claimiq_products
USING DELTA
AS
SELECT *
FROM products_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_claimiq_products;

In [ ]:
%sql
SELECT *
FROM bronze_claimiq_products
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM products_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_claimiq_products) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM products_source)
       = (SELECT COUNT(*) FROM bronze_claimiq_products)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

## 5.3 Providers - `providers.csv` -> `bronze_claimiq_providers`

**Business key:** `provider_id`. **What to inspect after each cell:** column
names match the Data Dictionary; the source and Bronze counts reconcile;
technical metadata columns are fully populated.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW providers_source
(
  source_record_id STRING,
  provider_id STRING,
  provider_type STRING,
  provider_region STRING,
  network_tier STRING,
  active_flag STRING,
  onboarding_date STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/claimiq/providers.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM providers_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE providers_source;

In [ ]:
%sql
SELECT COUNT(*) AS providers_source_count
FROM providers_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW providers_with_metadata AS
SELECT
  s.*,
  'providers.csv' AS _source_file_name,
  '/Volumes/workspace/default/claimiq/providers.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version
FROM providers_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW providers_bronze_ready AS
SELECT
  s.* EXCEPT (_corrupt_record),
  sha2(concat_ws('||',
    coalesce(cast(s.source_record_id AS STRING), '<NULL>'),
    coalesce(cast(s.provider_id AS STRING), '<NULL>'),
    coalesce(cast(s.provider_type AS STRING), '<NULL>'),
    coalesce(cast(s.provider_region AS STRING), '<NULL>'),
    coalesce(cast(s.network_tier AS STRING), '<NULL>'),
    coalesce(cast(s.active_flag AS STRING), '<NULL>'),
    coalesce(cast(s.onboarding_date AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM providers_with_metadata s;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_claimiq_providers
USING DELTA
AS
SELECT *
FROM providers_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_claimiq_providers;

In [ ]:
%sql
SELECT *
FROM bronze_claimiq_providers
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM providers_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_claimiq_providers) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM providers_source)
       = (SELECT COUNT(*) FROM bronze_claimiq_providers)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

## 5.4 Claim payments - `claim_payments.csv` -> `bronze_claimiq_claim_payments`

**Business key:** `payment_id`. **What to inspect after each cell:** column
names match the Data Dictionary; the source and Bronze counts reconcile;
technical metadata columns are fully populated.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_payments_source
(
  source_record_id STRING,
  payment_id STRING,
  claim_id STRING,
  payment_sequence STRING,
  payment_date STRING,
  payment_status STRING,
  payment_method STRING,
  paid_amount STRING,
  currency_code STRING,
  provider_id STRING,
  source_system STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/claimiq/claim_payments.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT *
FROM claim_payments_source
LIMIT 10;

In [ ]:
%sql
DESCRIBE claim_payments_source;

In [ ]:
%sql
SELECT COUNT(*) AS claim_payments_source_count
FROM claim_payments_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_payments_with_metadata AS
SELECT
  s.*,
  'claim_payments.csv' AS _source_file_name,
  '/Volumes/workspace/default/claimiq/claim_payments.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version
FROM claim_payments_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_payments_bronze_ready AS
SELECT
  s.* EXCEPT (_corrupt_record),
  sha2(concat_ws('||',
    coalesce(cast(s.source_record_id AS STRING), '<NULL>'),
    coalesce(cast(s.payment_id AS STRING), '<NULL>'),
    coalesce(cast(s.claim_id AS STRING), '<NULL>'),
    coalesce(cast(s.payment_sequence AS STRING), '<NULL>'),
    coalesce(cast(s.payment_date AS STRING), '<NULL>'),
    coalesce(cast(s.payment_status AS STRING), '<NULL>'),
    coalesce(cast(s.payment_method AS STRING), '<NULL>'),
    coalesce(cast(s.paid_amount AS STRING), '<NULL>'),
    coalesce(cast(s.currency_code AS STRING), '<NULL>'),
    coalesce(cast(s.provider_id AS STRING), '<NULL>'),
    coalesce(cast(s.source_system AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM claim_payments_with_metadata s;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_claimiq_claim_payments
USING DELTA
AS
SELECT *
FROM claim_payments_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_claimiq_claim_payments;

In [ ]:
%sql
SELECT *
FROM bronze_claimiq_claim_payments
LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM claim_payments_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_claimiq_claim_payments) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM claim_payments_source)
       = (SELECT COUNT(*) FROM bronze_claimiq_claim_payments)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

# Part 6 - Reconcile all sources and prove safe repeat-run behaviour

## 6.1 All six tables in one place

In [ ]:
%sql
SHOW TABLES LIKE 'bronze_claimiq_*';

## 6.2 Source-vs-Bronze reconciliation for every source

**Checkpoint:** every row must show `status = 'PASS'` before Week 4 can be
closed. `count_difference` of anything other than `0` means a source was
filtered, mis-pathed, or partially loaded.

In [ ]:
%sql
WITH counts AS (
  SELECT 'claims' AS dataset,
         (SELECT COUNT(*) FROM claims_source) AS source_count,
         (SELECT COUNT(*) FROM bronze_claimiq_claims) AS bronze_count
  UNION ALL
  SELECT 'policies',
         (SELECT COUNT(*) FROM policies_source),
         (SELECT COUNT(*) FROM bronze_claimiq_policies)
  UNION ALL
  SELECT 'policyholders',
         (SELECT COUNT(*) FROM policyholders_source),
         (SELECT COUNT(*) FROM bronze_claimiq_policyholders)
  UNION ALL
  SELECT 'products',
         (SELECT COUNT(*) FROM products_source),
         (SELECT COUNT(*) FROM bronze_claimiq_products)
  UNION ALL
  SELECT 'providers',
         (SELECT COUNT(*) FROM providers_source),
         (SELECT COUNT(*) FROM bronze_claimiq_providers)
  UNION ALL
  SELECT 'claim_payments',
         (SELECT COUNT(*) FROM claim_payments_source),
         (SELECT COUNT(*) FROM bronze_claimiq_claim_payments)
)
SELECT
  *,
  bronze_count - source_count AS count_difference,
  CASE WHEN source_count = bronze_count THEN 'PASS'
       ELSE 'INVESTIGATE' END AS status
FROM counts;

## 6.3 Lineage completeness check

**Pass condition:** every `missing_*` column is `0` for every source. A
non-zero value means the Bronze-ready view was built or joined incorrectly
before the write.

In [ ]:
%sql
SELECT 'claims' AS dataset,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_claimiq_claims
UNION ALL
SELECT 'policies',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_claimiq_policies
UNION ALL
SELECT 'policyholders',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_claimiq_policyholders
UNION ALL
SELECT 'products',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_claimiq_products
UNION ALL
SELECT 'providers',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_claimiq_providers
UNION ALL
SELECT 'claim_payments',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_claimiq_claim_payments;

## 6.4 Rescued-record visibility

This does not need to be zero - it only needs to be **inspected**. Any row
here is a record Spark could not parse under the approved schema; it was
retained, not silently dropped, per the raw-preserving Bronze contract.

In [ ]:
%sql
SELECT 'policies' AS dataset, COUNT(*) AS rows_with_rescued_context
FROM bronze_claimiq_policies
WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'policyholders', COUNT(*)
FROM bronze_claimiq_policyholders
WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'products', COUNT(*)
FROM bronze_claimiq_products
WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'providers', COUNT(*)
FROM bronze_claimiq_providers
WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'claim_payments', COUNT(*)
FROM bronze_claimiq_claim_payments
WHERE _rescued_payload IS NOT NULL;

## 6.5 Prove the repeat-run scenario does not duplicate

**Scenario (from the Project Playbook):** a rerun appends the same source
twice because Bronze has no batch identity or overwrite/merge rule.

**Rerun strategy used here:** every Bronze table in this notebook is written
with `CREATE OR REPLACE TABLE ... AS SELECT`, which always produces a full,
deterministic snapshot of the current source read rather than appending to
the previous one. This makes the load idempotent by construction: running
the same controlled batch twice can never double the row count.

**Instruction:** without changing any source file, re-run every code cell in
Parts 1-5 from top to bottom (or use **Run all**), then run the two cells
below.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM bronze_claimiq_claims) AS claims_after_rerun,
  (SELECT COUNT(*) FROM bronze_claimiq_policies) AS policies_after_rerun,
  (SELECT COUNT(*) FROM bronze_claimiq_policyholders) AS policyholders_after_rerun,
  (SELECT COUNT(*) FROM bronze_claimiq_products) AS products_after_rerun,
  (SELECT COUNT(*) FROM bronze_claimiq_providers) AS providers_after_rerun,
  (SELECT COUNT(*) FROM bronze_claimiq_claim_payments) AS claim_payments_after_rerun;

**Checkpoint:** every `*_after_rerun` value above must equal the
`bronze_count` recorded the first time this notebook ran. If any value is
larger, the write strategy silently appended instead of replacing - stop and
correct it before continuing to Week 5.

## 6.6 Confirm the rerun in Delta history

`DESCRIBE HISTORY` shows one additional `WRITE`/`CREATE OR REPLACE TABLE`
operation per rerun, but the table's row count does not grow - this is the
evidence a mentor will check.

In [ ]:
%sql
DESCRIBE HISTORY bronze_claimiq_claims;

In [ ]:
%sql
DESCRIBE HISTORY bronze_claimiq_policies;

In [ ]:
%sql
DESCRIBE HISTORY bronze_claimiq_policyholders;

In [ ]:
%sql
DESCRIBE HISTORY bronze_claimiq_products;

In [ ]:
%sql
DESCRIBE HISTORY bronze_claimiq_providers;

In [ ]:
%sql
DESCRIBE HISTORY bronze_claimiq_claim_payments;

# Part 7 - Inspect the pipeline result and capture evidence

1. Open **Catalog Explorer** and confirm all six tables exist under
   `workspace.default`: `bronze_claimiq_claims`, `bronze_claimiq_policies`,
   `bronze_claimiq_policyholders`, `bronze_claimiq_products`,
   `bronze_claimiq_providers`, `bronze_claimiq_claim_payments`.
2. Open each table's **Details** and **Sample Data** tabs and confirm the
   business columns plus the seven technical columns are visible.
3. Save screenshots of: the Volume listing, the reconciliation query result
   (Part 6.2, all `PASS`), the lineage completeness result (Part 6.3, all
   zero), and the rerun proof (Part 6.5, unchanged counts) into
   `screenshots/`.
4. Update `docs/pipeline_walkthrough.md` and `weekly_logs/week04_log.md`
   with what was built, what was checked, and what each teammate can
   explain.
5. Commit with a message such as
   `feat: ingest ClaimIQ sources to raw-preserving Bronze`.

# Part 8 - Troubleshooting and recovery

| Problem | Likely cause | Recovery |
|---|---|---|
| File not found | path, catalog, schema, Volume or filename mismatch | copy the exact path from the Part 1.2 Volume listing |
| Unexpected columns | wrong file or reader options | compare with the Data Dictionary and Week 3 schema |
| CSV appears as one column | wrong delimiter or header setting | confirm `header 'true'` and the approved CSV options |
| JSON fields are missing | incorrect JSON layout assumption | confirm JSON Lines (`multiLine 'false'`) versus multi-line JSON |
| Bronze count differs from source | wrong source path, filtering, or partial rerun | stop and compare the source view, the Bronze-ready view and the table |
| Duplicate increase after rerun | `INSERT`/`MERGE` append used instead of `CREATE OR REPLACE TABLE` | restore the approved full-replace write pattern from Parts 3-5 |
| Technical metadata is null | Bronze-ready view was not built from the metadata view correctly | inspect the `_with_metadata` view before writing the table |
| Table not visible in Catalog Explorer | wrong catalog/schema or insufficient access | confirm the active context from Part 1.1 and permissions |
| `claims_source` load fails on Parquet options | CSV-style options (`header`, `mode`) passed to a Parquet reader | remove CSV-only options; Parquet needs only `path` |

After a correction, rerun from the source-view cell for that dataset and
repeat all of its checks.

# Part 9 - Week 4 exit checklist

The team is ready to close Week 4 only when every statement is true:

- [ ] All six approved ClaimIQ batch sources are listed in the source inventory.
- [ ] Every source file is readable from its exact `/Volumes/workspace/default/claimiq` path.
- [ ] One correctly named `bronze_claimiq_*` Delta table exists per source.
- [ ] Source business values are preserved without any Silver transformation.
- [ ] Required technical metadata (`_source_file_name`, `_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_schema_version`, `_record_hash`) is present on every row.
- [ ] Rescued/corrupt context is retained where applicable (CSV/JSON sources).
- [ ] Source and Bronze counts reconcile for every one of the six sources.
- [ ] The controlled repeat-run test (Part 6.5-6.6) does not create unintended duplicates.
- [ ] Catalog Explorer inspection is complete for all six tables.
- [ ] Required screenshots and evidence references are recorded in `screenshots/`.
- [ ] `weekly_logs/week04_log.md`, `docs/pipeline_walkthrough.md` and the AI Transparency Note are updated.
- [ ] Student A, B and C can each explain the full six-source Source-to-Bronze flow and their owned deep area (Bronze / metadata-rerun / reconciliation).
- [ ] No Week 5 or later implementation (typing, standardisation, joins, Silver/Gold, Power BI, streaming) has been added to this notebook.

**Viva question:** why is the Week 04 ClaimIQ output not trustworthy until
reconciliation and the repeat-run proof both pass for all six sources?

**System state after Week 4:** six Bronze batch tables
(`bronze_claimiq_claims`, `bronze_claimiq_policies`,
`bronze_claimiq_policyholders`, `bronze_claimiq_products`,
`bronze_claimiq_providers`, `bronze_claimiq_claim_payments`) retaining all
physical records and lineage; a proven rerun does not double any batch.